In [1]:
import os
import shutil

# 1. Clean & Clone
os.chdir('/kaggle/working')
if os.path.exists('Patch-DM'):
    shutil.rmtree('Patch-DM')
os.system('git clone https://github.com/mlpc-ucsd/Patch-DM.git')
os.chdir('/kaggle/working/Patch-DM')

# 2. Installs
print("📦 Installing dependencies...")
os.system('pip install -q pytorch-lightning==1.9.5 lmdb==1.2.1 lpips pytorch-fid einops')
os.system('pip install -q git+https://github.com/openai/CLIP.git')

# 3. CRITICAL: Re-inject the Novelty Architecture
print("🧬 Injecting GatedWeightBlender into UNet...")
with open('model/unet.py', 'r') as f:
    unet_lines = f.readlines()

blender_class = [
    "\nimport torch.nn as nn\n", 
    "class GatedWeightBlender(nn.Module):\n",
    "    def __init__(self, channels):\n",
    "        super().__init__()\n",
    "        self.gate = nn.Sequential(\n",
    "            nn.Conv2d(channels, channels // 2, kernel_size=1),\n",
    "            nn.SiLU(),\n",
    "            nn.Conv2d(channels // 2, 1, kernel_size=1),\n",
    "            nn.Sigmoid()\n",
    "        )\n",
    "    def forward(self, x_orig, x_shifted):\n",
    "        alpha = self.gate(x_orig)\n",
    "        return alpha * x_orig + (1 - alpha) * x_shifted\n\n"
]
unet_lines.insert(15, "".join(blender_class))
unet_content = "".join(unet_lines)

# Inject initialization 
init_anchor = "if conf.resnet_use_zero_module:"
unet_content = unet_content.replace(init_anchor, f"self.blender = GatedWeightBlender(ch)\n        {init_anchor}", 1)

# Inject blending into forward pass
forward_anchor = "pred = self.out(h)"
blending_logic = "h_blended = self.blender(h, torch.roll(h, shifts=(2,2), dims=(2,3)))\n        pred = self.out(h_blended)"
unet_content = unet_content.replace(forward_anchor, blending_logic, 1)

with open('model/unet.py', 'w') as f:
    f.write(unet_content)

# 4. Standard Kaggle Inference Patches (With 1000-Image Limit!)
print("🔧 Patching legacy code and setting 1,000-image limit...")
with open('experiment.py', 'r') as f: exp_content = f.read()
exp_content = exp_content.replace('from numpy.lib.function_base import flip', 'from numpy import flip')
with open('experiment.py', 'w') as f: f.write(exp_content)

with open('test.py', 'r') as f: test_content = f.read()
if 'if i >= 250: break' not in test_content:
    test_content = test_content.replace(
        'for i, batch in enumerate(generator):', 
        'for i, batch in enumerate(generator):\n        if i >= 250: break  # 1000-IMAGE LIMIT'
    )
with open('test.py', 'w') as f: f.write(test_content)

print("✅ Custom architecture built! Ready for generation.")

Cloning into 'Patch-DM'...


📦 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
🧬 Injecting GatedWeightBlender into UNet...
🔧 Patching legacy code and setting 1,000-image limit...
✅ Custom architecture built! Ready for generation.


In [2]:
!python test.py \
  --batch_size 4 \
  --patch_size 64 \
  --image_size 256x256 \
  --output_dir /kaggle/working/novelty_faces \
  --full_path /kaggle/input/datasets/pragatirokade/novelty-weights/novelty_last.ckpt

conf: 
Global seed set to 0
Model params: 119.70 M
==Model size is 64==
ckpt path: /kaggle/input/datasets/pragatirokade/novelty-weights/novelty_last.ckpt
resume!
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=[0])` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=[0])` instead.
  rank_zero_deprecation(
Using 16bit None Automatic Mixed Precision (AMP)
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/plugins/precision/native_amp.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/checkpoint_connector.py:55: LightningDeprecationWarning: Setting `Trainer(resume_from_checkpoint=)` is deprecated in v1.5 and will be removed in v2.0.

In [1]:
import os
import glob

output_dir = "/kaggle/input/datasets/pragatirokade/novel-generated/novelty_faces"
if os.path.exists(output_dir):
    num_fakes = len(glob.glob(f"{output_dir}/**/*.png", recursive=True))
    print(f"🎉 You successfully generated {num_fakes} novelty faces!")
else:
    print("Folder not found. Double check the output path!")

🎉 You successfully generated 4064 novelty faces!


In [2]:
# 1. Ensure the library is installed on this new T4 session
!pip install -q pytorch-fid

import os
import glob

# Your newly generated faces
FAKE_PATH = "/kaggle/input/datasets/pragatirokade/novel-generated/novelty_faces"

# The base path to your real dataset
base_real_path = "/kaggle/input/datasets/denislukovnikov/ffhq256-images-only"
REAL_PATH = None

# 2. Automatically hunt down the exact subfolder holding the real images
print(f"🔍 Searching for the exact real image folder...")
for root, dirs, files in os.walk(base_real_path):
    if any(f.endswith(('.png', '.jpg', '.jpeg')) for f in files):
        REAL_PATH = root
        break

if not REAL_PATH:
    print(f"❌ Could not find any .png or .jpg files inside {base_real_path}. Check your dataset path!")
else:
    # Count everything to be 100% sure
    num_fakes = len(glob.glob(f"{FAKE_PATH}/**/*.png", recursive=True))
    num_reals = len([f for f in os.listdir(REAL_PATH) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"✅ Real images found: {num_reals} (Located at: {REAL_PATH})")
    print(f"✅ Fake novelty images found: {num_fakes}")
    print(f"📊 Firing up Inception network for the FINAL SCORE...\n")
    
    # 3. Run the official FID calculation
    !python -m pytorch_fid {REAL_PATH} {FAKE_PATH} --device cuda:0

🔍 Searching for the exact real image folder...
✅ Real images found: 70000 (Located at: /kaggle/input/datasets/denislukovnikov/ffhq256-images-only/ffhq256)
✅ Fake novelty images found: 4064
📊 Firing up Inception network for the FINAL SCORE...

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100%|███████████████████████████████████████| 91.2M/91.2M [00:00<00:00, 262MB/s]
100%|███████████████████████████████████████████| 82/82 [00:18<00:00,  4.37it/s]
FID:  397.49471738283637
